In [ ]:
from pathlib import Path

import cfgrib
import numpy as np
import xarray as xr
import xesmf as xe

In [ ]:
ds_ec = xr.open_dataset(Path.home() / "ml-ds_data" / "EC-Earth3.grid.nc")

In [ ]:
ds_era5_pl = xr.open_dataset(
    sorted((Path.home() / "ml-ds_data" / "ERA5" / "2011" / "pressure-levels").glob("*.grib"))[0]
)

In [ ]:
ds_era5_sl = xr.open_dataset(
    sorted((Path.home() / "ml-ds_data" / "ERA5" / "2011" / "single-levels").glob("*.grib"))[0],
    engine="cfgrib",
    backend_kwargs={"filter_by_keys": {"stepType": "instant"}},
)

In [ ]:
carra_path = sorted((Path.home() / "ml-ds_data" / "CARRA2" / "2011").glob("*.grib"))[0]
ds_carra2 = cfgrib.open_datasets(carra_path)
ds_carra2_vars = sorted({name for ds in ds_carra2 for name in ds.data_vars})

In [ ]:
ds_carra2_t2m = next(ds for ds in ds_carra2 if "t2m" in ds)

era_time_dim = "valid_time" if "valid_time" in ds_era5_sl.t2m.dims else "time"
carra_time_dim = "valid_time" if "valid_time" in ds_carra2_t2m.t2m.dims else "time"

t2m_era = ds_era5_sl.t2m.isel({era_time_dim: 0})
t2m_carra = ds_carra2_t2m.t2m.isel({carra_time_dim: 0})

In [ ]:
coarse_lon = ds_ec["lon"].values
# Shift longitudes to the range [-180, 180] - for era5
coarse_lon = ((coarse_lon + 180) % 360) - 180
coarse_lon = np.sort(coarse_lon)
coarse_lat = ds_ec["lat"].values

In [ ]:
# Interpolate ERA5 -> EC-Earth3 grid (coarse)
t2m_era_coarse = t2m_era.interp(
    latitude=coarse_lat,
    longitude=coarse_lon,
    method="linear",
)
t2m_era_coarse = t2m_era_coarse.dropna(dim="latitude", how="all").dropna(dim="longitude", how="all")

In [ ]:
t2m_era.plot()

In [ ]:
t2m_era_coarse.plot()

In [ ]:
# Regrid ERA5 coarse t2m -> CARRA curvilinear grid
def _pick_coord(ds, candidates):
    for name in candidates:
        if name in ds.coords:
            return ds.coords[name]
        if name in ds:
            return ds[name]
    raise KeyError(f"None of {candidates} found in dataset")

carra_lon = _pick_coord(ds_carra2_t2m, ["longitude", "lon"])
carra_lat = _pick_coord(ds_carra2_t2m, ["latitude", "lat"])

In [ ]:
# Cut out Norway
# carra_lon = carra_lon.isel(x=slice(2100, 2500), y=slice(400, 1500))
# carra_lat = carra_lat.isel(x=slice(2100, 2500), y=slice(400, 1500))

In [ ]:
# xESMF expects source grid names lon/lat
src = t2m_era_coarse.rename({"longitude": "lon", "latitude": "lat"})

# Use lazy dask chunks to avoid allocating the full regridded time stack in memory
if "valid_time" in src.dims:
    src = src.chunk({"valid_time": 1})

# Curvilinear target grid can be provided as 2D lon/lat arrays
grid_out = xr.Dataset({"lon": carra_lon, "lat": carra_lat})

In [ ]:
regridder = xe.Regridder(src, grid_out, method="bilinear", periodic=False, reuse_weights=False)
era5_t2m_on_carra = regridder(src, keep_attrs=True, output_chunks={"valid_time": 1}, skipna=True)

In [ ]:
era5_t2m_on_carra.plot(size=10)

In [ ]:
t2m_carra.plot(size=10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(ncols=2, figsize=(8, 11), constrained_layout=True)

era5_t2m_on_carra.isel(x=slice(2100, 2500), y=slice(400, 1400)).plot(ax=axes[0])
axes[0].set_title("ERA5 t2m on CARRA")

t2m_carra.isel(x=slice(2100, 2500), y=slice(400, 1400)).plot(ax=axes[1])
axes[1].set_title("CARRA t2m")